In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Reuse variable names across steps for efficiency
tz = pytz.timezone('Asia/Shanghai')

# 1) Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')

# Ensure 'value' is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Helper to aggregate a dataframe given the name column and source label
def aggregate_df(df, name_col, source_label):
    # Group and compute count, min/max timestamps, and global P95 of value
    grouped = df.groupby(['cmdb_id', name_col]).agg(
        count=('value', 'count'),
        t_min=('timestamp', 'min'),
        t_max=('timestamp', 'max'),
        p95_value=('value', lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size>0 else np.nan)
    ).reset_index()

    # Convert timestamps to UTC then to Asia/Shanghai and format compactly
    grouped['t_min'] = pd.to_datetime(grouped['t_min'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    grouped['t_max'] = pd.to_datetime(grouped['t_max'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    # Round p95 for compact display
    grouped['p95_value'] = grouped['p95_value'].round(6)

    # Normalize name column to 'kpi_name' for combined table
    grouped = grouped.rename(columns={name_col: 'kpi_name'})
    grouped.insert(0, 'source', source_label)
    return grouped

# 2-3) Aggregate each file
agg_metric = aggregate_df(df_metric, 'kpi_name', 'metric')
agg_trace = aggregate_df(df_trace, 'trace_name', 'trace')
agg_log = aggregate_df(df_log, 'log_name', 'log')

# 4) Combine tables
combined = pd.concat([agg_metric, agg_trace, agg_log], ignore_index=True, sort=False)

# Sort by count descending and limit to top 50 rows
combined_sorted_top50 = combined.sort_values(by='count', ascending=False).head(50).reset_index(drop=True)

# 5) Distinct cmdb_id lists per file (compact series)
cmdb_metric = pd.Series(sorted(df_metric['cmdb_id'].dropna().unique()), name='cmdb_id_metric')
cmdb_trace = pd.Series(sorted(df_trace['cmdb_id'].dropna().unique()), name='cmdb_id_trace')
cmdb_log = pd.Series(sorted(df_log['cmdb_id'].dropna().unique()), name='cmdb_id_log')

# Final displayed outputs (compact)
combined_sorted_top50, cmdb_metric, cmdb_trace, cmdb_log

```
Out[1]:
```
import pandas as pd

# Reuse variables from previous execution:
# combined_sorted_top50, cmdb_metric, cmdb_trace, cmdb_log

# Prepare compact summary details
total_rows = combined_sorted_top50.shape[0]
source_counts = combined_sorted_top50['source'].value_counts().to_dict()

# Distinct cmdb_id lists and counts
metric_list = ", ".join(cmdb_metric.tolist())
trace_list = ", ".join(cmdb_trace.tolist())
log_list = ", ".join(cmdb_log.tolist())
metric_count = len(cmdb_metric)
trace_count = len(cmdb_trace)
log_count = len(cmdb_log)

# Top 5 groups by count for quick reference
top5 = combined_sorted_top50[['source','cmdb_id','kpi_name','count','p95_value']].head(5)
top5_lines = "; ".join([
    f"{i+1}) {row.source} | {row.cmdb_id} | {row.kpi_name} | count={int(row['count'])} | p95={row.p95_value}"
    for i, row in enumerate(top5.itertuples(index=False))
])

summary = (
    f"Summary of analysis:\n"
    f"- Combined table: top {total_rows} groups (sorted by count desc).\n"
    f"- Source breakdown (in top {total_rows}): " +
    ", ".join([f"{k}: {v}" for k, v in source_counts.items()]) + ".\n"
    f"- Distinct cmdb_id per file: metric ({metric_count}): [{metric_list}]; "
    f"trace ({trace_count}): [{trace_list}]; log ({log_count}): [{log_list}].\n"
    f"- Top 5 groups by count: {top5_lines}.\n"
    f"Notes: The combined table was limited to the top 50 groups; full per-group details are available in the combined_sorted_top50 variable."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(    source       cmdb_id                                           kpi_name  count                    t_min                    t_max      p95_value
0   metric          IG01                            OSLinux-CPU_CPU_CPULoad     31  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST       1.110000
1   metric          IG01                         OSLinux-CPU_CPU_CPUSysTime     31  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST       0.490618
2   metric          IG01                        OSLinux-CPU_CPU_CPUUserTime     31  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST      25.285350
3   metric          IG01                             OSLinux-CPU_CPU_CPUWio     31  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST       0.264000
4   metric       Redis02                            OSLinux-CPU_CPU_CPULoad     31  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST       0.080000
..     ...           ...                                                ...    ...                      ...                      ...            ...
45  metric          IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST       3.000000
46  metric      Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST       0.000000
47  metric      Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST     927.000000
48  metric      Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST  405396.000000
49  metric  ServiceTest9                                            app.mrt     31  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST       0.467150

[50 rows x 7 columns], 0         IG01
1         IG02
2         MG01
3         MG02
4      Mysql01
        ...   
24    apache02
25    dockerA1
26    dockerA2
27    dockerB1
28    dockerB2
Name: cmdb_id_metric, Length: 29, dtype: object, 0         IG01
1         IG02
2         MG01
3         MG02
4     Tomcat01
5     Tomcat02
6     Tomcat03
7     Tomcat04
8     dockerA1
9     dockerA2
10    dockerB1
11    dockerB2
Name: cmdb_id_trace, dtype: object, 0    Tomcat01
1    Tomcat02
2    Tomcat03
3    Tomcat04
4    apache01
5    apache02
Name: cmdb_id_log, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    
# Ensure numeric values
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P90 per group (cmdb_id, kpi_name) using entire series BEFORE filtering
def pct_func(q):
    return lambda x: float(np.nanpercentile(x.dropna(), q)) if x.dropna().size > 0 else np.nan

thresholds = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    p95_value=pct_func(95),
    p90_value=pct_func(90)
).reset_index()

# Merge thresholds back to metric rows for comparison
dfm = df_metric.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# 2) Filter rows to incident time window (inclusive) in UTC+8:
# Window: March 9, 2021 19:30:00 through 2021-03-09 20:00:00 Asia/Shanghai (UTC+8)
start_local = tz.localize(pd.Timestamp('2021-03-09 19:30:00'))
end_local   = tz.localize(pd.Timestamp('2021-03-09 20:00:00'))

# Convert to UTC epoch seconds for comparison with stored timestamps (which are epoch seconds)
start_utc_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_utc_ts   = int(end_local.astimezone(pytz.utc).timestamp())

df_win = dfm[(dfm['timestamp'] >= start_utc_ts) & (dfm['timestamp'] <= end_utc_ts)].copy()

# 3) Mark anomalies using global P95 (do not recompute thresholds after filtering)
df_win['is_anom_p95'] = (df_win['value'] >= df_win['p95_value']) & (~df_win['p95_value'].isna())
df_win['is_anom_p90'] = (df_win['value'] >= df_win['p90_value']) & (~df_win['p90_value'].isna())

# Helper to build anomaly summary given a boolean anomaly column name
def build_anom_summary(df_window, anom_col, threshold_col_name):
    if df_window.empty:
        return pd.DataFrame()  # no rows in window at all
    anom_rows = df_window[df_window[anom_col]]
    if anom_rows.empty:
        return pd.DataFrame()
    # Aggregations within window per group
    count_anom = anom_rows.groupby(['cmdb_id','kpi_name'])['value'].count().rename('count_anom')
    first_anom_ts = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].min().rename('first_anom_ts')
    max_value_in_window = df_window.groupby(['cmdb_id','kpi_name'])['value'].max().rename('max_value_in_window')
    t_min_window = df_window.groupby(['cmdb_id','kpi_name'])['timestamp'].min().rename('t_min_window')
    t_max_window = df_window.groupby(['cmdb_id','kpi_name'])['timestamp'].max().rename('t_max_window')
    # Compose
    summary = pd.concat([count_anom, first_anom_ts, max_value_in_window, t_min_window, t_max_window], axis=1).reset_index()
    # Attach threshold (p95 or p90) - take from thresholds DF
    summary = summary.merge(thresholds[['cmdb_id','kpi_name', threshold_col_name]],
                            on=['cmdb_id','kpi_name'], how='left')
    # Format timestamps to Asia/Shanghai strings
    for col in ['first_anom_ts','t_min_window','t_max_window']:
        summary[col] = pd.to_datetime(summary[col], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # Round numeric columns for compactness
    summary['p_threshold'] = summary[threshold_col_name].round(6)
    summary['max_value_in_window'] = summary['max_value_in_window'].round(6)
    # Select and rename columns to match requested output
    summary = summary[['cmdb_id','kpi_name','count_anom','first_anom_ts','max_value_in_window','p_threshold','t_min_window','t_max_window']]
    summary = summary.rename(columns={'p_threshold':'p95_value' if threshold_col_name=='p95_value' else 'p90_value'})
    return summary

# 4) Build P95 result
anom_p95_summary = build_anom_summary(df_win, 'is_anom_p95', 'p95_value')

# If no anomalies at P95, repeat using P90
used_threshold = 'p95'
if anom_p95_summary.empty:
    anom_p90_summary = build_anom_summary(df_win, 'is_anom_p90', 'p90_value')
    final_anom = anom_p90_summary.copy()
    used_threshold = 'p90'
else:
    final_anom = anom_p95_summary.copy()

# 4 continued: limit to top 20 by count_anom desc
if not final_anom.empty:
    final_anom_sorted = final_anom.sort_values(by='count_anom', ascending=False).head(20).reset_index(drop=True)
else:
    final_anom_sorted = final_anom  # empty DF

# 5) Aggregates: total number of metric groups that had anomalies, and distinct cmdb_id list (limit 20 entries)
total_groups_with_anom = int(final_anom.shape[0])
cmdb_list_involved = list(final_anom['cmdb_id'].drop_duplicates().astype(str).head(20))

# Final compact outputs
final_anom_sorted, total_groups_with_anom, cmdb_list_involved, used_threshold

```
Out[2]:
```
```python
# Compose a concise plain-English summary string from previous analysis variables
try:
    final_df = final_anom_sorted  # from previous execution
    total = total_groups_with_anom
    cmdbs = cmdb_list_involved
    thresh = used_threshold
except NameError:
    final_df = None
    total = 0
    cmdbs = []
    thresh = 'p95'

if final_df is None or final_df.shape[0] == 0:
    summary = ("No anomalies detected in the specified incident window using the available thresholds. "
               "Either there were no metric rows in the window or no values exceeded P95/P90.")
else:
    # Build top 5 lines
    top5_rows = []
    display_threshold_col = 'p95_value' if 'p95_value' in final_df.columns else 'p90_value'
    for i, row in enumerate(final_df.head(5).itertuples(index=False), start=1):
        cmdb = getattr(row, 'cmdb_id')
        kpi = getattr(row, 'kpi_name')
        cnt = int(getattr(row, 'count_anom'))
        first = getattr(row, 'first_anom_ts')
        maxv = float(getattr(row, 'max_value_in_window'))
        pval = float(getattr(row, display_threshold_col))
        top5_rows.append(f"{i}) {cmdb} | {kpi} | count={cnt} | first={first} | max={maxv:.6g} | {display_threshold_col}={pval:.6g}")
    top5_text = " ; ".join(top5_rows)

    summary = (
        f"Anomaly detection summary for metrics (window: 2021-03-09 19:30:00 to 2021-03-09 20:00:00 UTC+8):\n"
        f"- Threshold used: global {thresh.upper()} per metric group.\n"
        f"- Metric groups with anomalies (value >= threshold) in the window: {total} groups.\n"
        f"- Distinct cmdb_id involved (up to 20): {cmdbs}.\n"
        f"- Top 5 anomalous groups (cmdb_id | kpi_name | count_anom | first_anom_ts | max_value_in_window | {display_threshold_col}):\n  {top5_text}\n"
        f"Notes: Results are limited to the top groups by anomaly count; timestamps are shown in UTC+8."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count_anom            first_anom_ts  max_value_in_window     p95_value             t_min_window             t_max_window
0   Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
1   Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
2   Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
3   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...          31  2021-03-09 19:30:00 CST         3.300540e+05  3.300540e+05  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
4   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...          31  2021-03-09 19:30:00 CST         2.310000e+02  2.310000e+02  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
5       IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount          31  2021-03-09 19:30:00 CST         3.000000e+00  3.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
6       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax          31  2021-03-09 19:30:00 CST         2.040136e+09  2.040136e+09  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
7   Tomcat02    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
8   Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
9       IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies          31  2021-03-09 19:30:00 CST         1.000000e+00  1.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
10      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax          31  2021-03-09 19:30:00 CST         2.040136e+09  2.040136e+09  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
11  Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
12  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
13  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
14   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount          31  2021-03-09 19:30:00 CST         4.000000e+00  4.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
15   Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
16   Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
17  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...          31  2021-03-09 19:30:00 CST         5.000000e-01  5.000000e-01  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
18  apache01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST
19  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...          31  2021-03-09 19:30:00 CST         0.000000e+00  0.000000e+00  2021-03-09 19:30:00 CST  2021-03-09 20:00:00 CST, 1704, ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01'], 'p95')```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure numeric values
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Incident window in Asia/Shanghai (inclusive)
start_local = tz.localize(pd.Timestamp('2021-03-09 19:30:00'))
end_local   = tz.localize(pd.Timestamp('2021-03-09 20:00:00'))
start_utc_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_utc_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# Helper to compute global quantiles per group (do BEFORE filtering)
def compute_group_thresholds(df, q):
    name = f'p{int(q)}_value'
    agg = df.groupby(['cmdb_id','kpi_name'])['value'].agg(
        **{name: (lambda x, q=q: float(np.nanpercentile(x.dropna(), q)) if x.dropna().size>0 else np.nan)}
    ).reset_index()
    return agg

# Try to reuse previously computed thresholds if available
try:
    thresholds  # from prior runs
except NameError:
    thresholds = compute_group_thresholds(df_metric, 95).rename(columns={'p95_value':'p95_value'})  # ensure name

# Ensure both p95 and p90 are available in thresholds DataFrame
if 'p95_value' not in thresholds.columns:
    thresholds = compute_group_thresholds(df_metric, 95)
if 'p90_value' not in thresholds.columns:
    p90 = compute_group_thresholds(df_metric, 90).rename(columns={'p90_value':'p90_value'})
    # merge p90 into thresholds
    thresholds = thresholds.merge(p90, on=['cmdb_id','kpi_name'], how='left')

# Merge thresholds into metric rows (do not filter)
dfm = df_metric.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Filter to incident window rows
df_win = dfm[(dfm['timestamp'] >= start_utc_ts) & (dfm['timestamp'] <= end_utc_ts)].copy()

# Function to compute longest consecutive-minute run length given sorted integer timestamps (seconds)
def longest_consecutive_minute_run(ts_array):
    if len(ts_array) == 0:
        return 0
    # ensure sorted unique
    ts = np.sort(np.unique(ts_array.astype(np.int64)))
    if ts.size == 1:
        return 1
    diffs = np.diff(ts)
    # consecutive minutes means diff == 60
    # Build run segments where diff == 60
    runs = []
    run_len = 1
    for d in diffs:
        if d == 60:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return int(np.max(runs))

# Primary: use P95
threshold_col = 'p95_value'
anom_mask = (~df_win[threshold_col].isna()) & (df_win['value'] >= df_win[threshold_col])
anom_rows = df_win[anom_mask].copy()

def build_summary_for_threshold(df_window, anom_rows, threshold_col_name):
    if df_window.empty or anom_rows.empty:
        return pd.DataFrame()
    # count_anom and first_anom_ts from anom_rows
    grp_count = anom_rows.groupby(['cmdb_id','kpi_name'])['value'].count().rename('count_anom')
    grp_first = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].min().rename('first_anom_ts')
    # max_value_in_window should use all rows in window for the group
    grp_max = df_window.groupby(['cmdb_id','kpi_name'])['value'].max().rename('max_value_in_window')
    # p_threshold
    grp_thr = thresholds[['cmdb_id','kpi_name', threshold_col_name]].set_index(['cmdb_id','kpi_name'])
    # combine
    summary = pd.concat([grp_count, grp_first, grp_max], axis=1).reset_index()
    summary = summary.merge(grp_thr.reset_index(), on=['cmdb_id','kpi_name'], how='left')
    # compute severity_ratio with special rules
    def compute_ratio(row):
        p = row[threshold_col_name]
        mx = row['max_value_in_window']
        if pd.isna(p):
            return np.nan
        if p == 0:
            if mx > 0:
                return np.inf
            else:
                return 1.0
        else:
            return float(mx / p)
    summary['severity_ratio'] = summary.apply(compute_ratio, axis=1)
    # compute longest_consecutive_anom_run per group using anom_rows timestamps
    runs = []
    grouped_anom_ts = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp']
    for _, group in summary[['cmdb_id','kpi_name']].itertuples(index=False):
        # select timestamps array for this group
        try:
            arr = grouped_anom_ts.get_group((group[0], group[1])).values
        except Exception:
            # grouping keys order nuance: alternative access
            arr = anom_rows[(anom_rows['cmdb_id']==group[0]) & (anom_rows['kpi_name']==group[1])]['timestamp'].values
        run_len = longest_consecutive_minute_run(arr)
        runs.append(run_len)
    summary['longest_consecutive_anom_run'] = runs
    # format timestamps to UTC+8 strings
    summary['first_anom_ts'] = pd.to_datetime(summary['first_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # Round numeric outputs for compactness
    summary[threshold_col_name] = summary[threshold_col_name].round(6)
    summary['max_value_in_window'] = summary['max_value_in_window'].round(6)
    # significant breach boolean
    summary['significant_breach'] = summary['severity_ratio'].apply(lambda x: True if (pd.notna(x) and x >= 1.5) else False)
    # Select and order required columns
    out_col_name = 'p95_value' if threshold_col_name == 'p95_value' else 'p90_value'
    summary = summary.rename(columns={threshold_col_name: out_col_name})
    summary = summary[['cmdb_id','kpi_name', out_col_name, 'count_anom', 'first_anom_ts',
                       'max_value_in_window', 'severity_ratio', 'longest_consecutive_anom_run', 'significant_breach']]
    return summary

summary_p95 = build_summary_for_threshold(df_win, anom_rows, 'p95_value')

used_threshold = 'p95'
final_summary = summary_p95.copy()

# If no groups with p95 anomalies, retry with p90
if summary_p95.empty:
    # compute p90 anomalies
    threshold_col = 'p90_value'
    anom_mask90 = (~df_win[threshold_col].isna()) & (df_win['value'] >= df_win[threshold_col])
    anom_rows90 = df_win[anom_mask90].copy()
    summary_p90 = build_summary_for_threshold(df_win, anom_rows90, 'p90_value')
    final_summary = summary_p90.copy()
    used_threshold = 'p90'

# If still empty, final_summary will be empty DataFrame
if final_summary.empty:
    final_result = final_summary  # empty
    sig_count = 0
    sig_cmdbs = []
else:
    # compute severity sorting and pick top 20 by severity_ratio desc (put Inf at top)
    final_result = final_summary.sort_values(by='severity_ratio', ascending=False).head(20).reset_index(drop=True)
    # aggregates for significant breaches
    sig_df = final_result[final_result['significant_breach'] == True]
    sig_count = int(sig_df.shape[0])
    sig_cmdbs = sig_df['cmdb_id'].drop_duplicates().astype(str).head(20).tolist()

# Final compact outputs
final_result, sig_count, sig_cmdbs, used_threshold

```
Out[3]:
```
```python
# Build a concise plain-English summary from the previous analysis variables
try:
    df = final_result  # top 20 groups sorted by severity_ratio (from prior run)
    sig_count = sig_count
    sig_cmdbs = sig_cmdbs
    thresh = used_threshold
except NameError:
    df = None
    sig_count = 0
    sig_cmdbs = []
    thresh = 'p95'

if df is None or df.shape[0] == 0:
    summary = "No metric-group anomalies found in the incident window using available thresholds (P95/P90)."
else:
    # Prepare top 5 summary lines
    display_cols = ['cmdb_id','kpi_name','p95_value','count_anom','first_anom_ts','max_value_in_window','severity_ratio','longest_consecutive_anom_run','significant_breach']
    top5 = df.head(5)[display_cols]
    top5_lines = []
    for r in top5.itertuples(index=False):
        top5_lines.append(
            f"{r.cmdb_id} | {r.kpi_name} | p95={r.p95_value:.6g} | count={int(r.count_anom)} | first={r.first_anom_ts} | "
            f"max={r.max_value_in_window:.6g} | severity={r.severity_ratio:.6g} | run={int(r.longest_consecutive_anom_run)} | sig={bool(r.significant_breach)}"
        )
    top5_text = " ; ".join(top5_lines)

    summary = (
        f"Metric anomaly summary for window 2021-03-09 19:30:00 to 2021-03-09 20:00:00 (UTC+8):\n"
        f"- Threshold used: global {thresh.upper()} per group.\n"
        f"- Groups returned (top 20 by severity_ratio): {df.shape[0]} groups.\n"
        f"- Groups with significant breach (severity_ratio >= 1.5): {sig_count} groups.\n"
        f"- Distinct cmdb_id among significant groups (up to 20): {sig_cmdbs}.\n"
        f"- Top 5 groups by severity_ratio (cmdb_id | kpi_name | details):\n  {top5_text}\n"
        f"Notes: severity_ratio is max_value_in_window / p95_value; infinite severity occurs when p95_value == 0 and max>0."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  p95_value  count_anom            first_anom_ts  max_value_in_window  severity_ratio  longest_consecutive_anom_run  significant_breach
0       IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT   0.000000          29  2021-03-09 19:30:00 CST               1.0000             inf                             0                True
1       IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT   0.000000          31  2021-03-09 19:30:00 CST               1.0000             inf                             0                True
2       MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT   0.000000          21  2021-03-09 19:30:00 CST               1.0000             inf                             0                True
3   Tomcat02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT   0.000000          21  2021-03-09 19:30:00 CST               1.0000             inf                             0                True
4       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT   0.050000           1  2021-03-09 19:39:00 CST               1.0000       20.000000                             0                True
5    Mysql02      Mysql-MySQL_3306_Innodb os log pending fsyncs   0.200000           1  2021-03-09 19:52:00 CST               1.0000        5.000000                             0                True
6    Mysql02        Mysql-MySQL_3306_Innodb data pending fsyncs   0.200000           1  2021-03-09 19:52:00 CST               1.0000        5.000000                             0                True
7    Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead   5.600000           1  2021-03-09 20:00:00 CST              22.4000        4.000000                             0                True
8    Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps   0.050000           1  2021-03-09 20:00:00 CST               0.2000        4.000000                             0                True
9    Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...   0.840000           1  2021-03-09 19:33:00 CST               2.8000        3.333333                             0                True
10      IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps   0.600000           1  2021-03-09 19:59:00 CST               2.0000        3.333333                             0                True
11   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps   1.500000           1  2021-03-09 19:45:00 CST               5.0000        3.333333                             0                True
12  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps   1.800000           1  2021-03-09 19:55:00 CST               6.0000        3.333333                             0                True
13   Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages dirty  28.800000           1  2021-03-09 19:52:00 CST              90.0000        3.125000                             0                True
14      IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...   0.070000           1  2021-03-09 19:41:00 CST               0.2000        2.857143                             0                True
15      IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps   0.350000           1  2021-03-09 19:41:00 CST               1.0000        2.857143                             0                True
16   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead   9.533325           1  2021-03-09 19:34:00 CST              24.5333        2.573425                             0                True
17   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead   7.066667           1  2021-03-09 19:59:00 CST              16.4000        2.320755                             0                True
18   Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time   0.300800           1  2021-03-09 19:40:00 CST               0.6833        2.271609                             0                True
19   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps   0.331677           1  2021-03-09 19:59:00 CST               0.7167        2.160838                             0                True, 20, ['IG01', 'IG02', 'MG02', 'Tomcat02', 'MG01', 'Mysql02', 'Mysql01', 'apache02'], 'p95')```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing df_trace if available, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure numeric values
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Incident window in Asia/Shanghai (inclusive)
start_local = tz.localize(pd.Timestamp('2021-03-09 19:30:00'))
end_local   = tz.localize(pd.Timestamp('2021-03-09 20:00:00'))
start_utc_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_utc_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# Helper to compute global quantile per group BEFORE filtering
def compute_group_quantile(df, q, col_name):
    agg = df.groupby(['cmdb_id','trace_name'])['value'].agg(
        **{col_name: (lambda x, q=q: float(np.nanpercentile(x.dropna(), q)) if x.dropna().size>0 else np.nan)}
    ).reset_index()
    return agg

# Compute global P95 and P90 for trace groups
p95_df = compute_group_quantile(df_trace, 95, 'p95_value')
p90_df = compute_group_quantile(df_trace, 90, 'p90_value')
thresholds = p95_df.merge(p90_df, on=['cmdb_id','trace_name'], how='left')

# Merge thresholds into trace rows
df_t = df_trace.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

# Filter to incident window rows
df_win = df_t[(df_t['timestamp'] >= start_utc_ts) & (df_t['timestamp'] <= end_utc_ts)].copy()

# Helper: longest consecutive minute run from array of epoch-second timestamps
def longest_consecutive_minute_run(ts_array):
    if len(ts_array) == 0:
        return 0
    ts = np.sort(np.unique(ts_array.astype(np.int64)))
    run_len = 1
    runs = []
    for d in np.diff(ts):
        if d == 60:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return int(np.max(runs))

# Function to build summary given threshold column name ('p95_value' or 'p90_value')
def build_trace_anom_summary(df_window, thresholds_df, threshold_col):
    if df_window.empty:
        return pd.DataFrame()
    # Mask rows where value >= threshold (and threshold not NaN)
    mask = (~df_window[threshold_col].isna()) & (df_window['value'] >= df_window[threshold_col])
    anom_rows = df_window[mask].copy()
    if anom_rows.empty:
        return pd.DataFrame()
    # Aggregations
    grp_count = anom_rows.groupby(['cmdb_id','trace_name'])['value'].count().rename('count_anom')
    grp_first = anom_rows.groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('first_anom_ts')
    grp_max = df_window.groupby(['cmdb_id','trace_name'])['value'].max().rename('max_value_in_window')
    summary = pd.concat([grp_count, grp_first, grp_max], axis=1).reset_index()
    # attach threshold
    summary = summary.merge(thresholds_df[['cmdb_id','trace_name', threshold_col]],
                            on=['cmdb_id','trace_name'], how='left')
    # severity_ratio with special rules
    def compute_ratio(row):
        p = row[threshold_col]
        mx = row['max_value_in_window']
        if pd.isna(p):
            return np.nan
        if p == 0:
            if mx > 0:
                return np.inf
            else:
                return 1.0
        return float(mx / p)
    summary['severity_ratio'] = summary.apply(compute_ratio, axis=1)
    # longest consecutive run using anom_rows grouped timestamps
    runs = []
    grouped_ts = anom_rows.groupby(['cmdb_id','trace_name'])['timestamp']
    for cmdb, trace_name in summary[['cmdb_id','trace_name']].itertuples(index=False):
        try:
            arr = grouped_ts.get_group((cmdb, trace_name)).values
        except Exception:
            arr = anom_rows[(anom_rows['cmdb_id']==cmdb) & (anom_rows['trace_name']==trace_name)]['timestamp'].values
        runs.append(longest_consecutive_minute_run(arr))
    summary['longest_consecutive_anom_run'] = runs
    # format first_anom_ts to UTC+8 string
    summary['first_anom_ts'] = pd.to_datetime(summary['first_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # round numeric columns
    summary[threshold_col] = summary[threshold_col].round(6)
    summary['max_value_in_window'] = summary['max_value_in_window'].round(6)
    # select and order columns
    out_col = 'p95_value' if threshold_col == 'p95_value' else 'p90_value'
    summary = summary.rename(columns={threshold_col: out_col})
    summary = summary[['cmdb_id','trace_name', out_col, 'count_anom', 'first_anom_ts',
                       'max_value_in_window', 'severity_ratio', 'longest_consecutive_anom_run']]
    return summary

# Try P95 first
summary_p95 = build_trace_anom_summary(df_win, thresholds, 'p95_value')
used_threshold = 'p95'
final_summary = summary_p95.copy()

# If no anomalies at P95, try P90
if final_summary.empty:
    summary_p90 = build_trace_anom_summary(df_win, thresholds, 'p90_value')
    final_summary = summary_p90.copy()
    used_threshold = 'p90'

# Prepare final outputs: top 20 by severity_ratio desc
if final_summary.empty:
    final_trace_anom_top20 = final_summary  # empty
    total_anom_groups = 0
    anomalous_cmdbs = []
    freq_by_cmdb_top10 = pd.DataFrame(columns=['cmdb_id','anomalous_group_count'])
else:
    final_trace_anom_top20 = final_summary.sort_values(by='severity_ratio', ascending=False).head(20).reset_index(drop=True)
    total_anom_groups = int(final_summary.shape[0])
    anomalous_cmdbs = final_summary['cmdb_id'].drop_duplicates().astype(str).head(20).tolist()
    freq_by_cmdb = final_summary.groupby('cmdb_id').size().reset_index(name='anomalous_group_count').sort_values(by='anomalous_group_count', ascending=False)
    freq_by_cmdb_top10 = freq_by_cmdb.head(10).reset_index(drop=True)

# Compact outputs
final_trace_anom_top20, total_anom_groups, anomalous_cmdbs, freq_by_cmdb_top10, used_threshold

```
Out[4]:
```
```python
# Build a concise plain-English summary from the previous trace analysis variables
try:
    df_top20 = final_trace_anom_top20  # top 20 anomalous trace groups (from prior run)
    total = total_anom_groups
    cmdbs = anomalous_cmdbs
    freq_tbl = freq_by_cmdb_top10
    thresh = used_threshold
except NameError:
    df_top20 = None
    total = 0
    cmdbs = []
    freq_tbl = None
    thresh = 'p95'

if df_top20 is None or df_top20.shape[0] == 0:
    summary = "No trace-group anomalies detected in the incident window using available thresholds (P95/P90)."
else:
    # Prepare top 5 summary lines
    top5_lines = []
    for r in df_top20.head(5).itertuples(index=False):
        # r: cmdb_id, trace_name, p95_value/p90_value, count_anom, first_anom_ts, max_value_in_window, severity_ratio, longest_consecutive_anom_run
        pval = getattr(r, df_top20.columns[2])
        top5_lines.append(
            f"{r.cmdb_id} | {r.trace_name} | p{thresh.upper()}={pval:.6g} | count={int(r.count_anom)} | first={r.first_anom_ts} | "
            f"max={r.max_value_in_window:.6g} | severity={r.severity_ratio:.6g} | run={int(r.longest_consecutive_anom_run)}"
        )
    top5_text = " ; ".join(top5_lines)

    summary = (
        f"Trace anomaly summary for window 2021-03-09 19:30:00 to 2021-03-09 20:00:00 (UTC+8):\n"
        f"- Threshold used: global {thresh.upper()} per trace group.\n"
        f"- Total trace groups with anomalies in window: {total}.\n"
        f"- Distinct cmdb_id involved (up to 20): {cmdbs}.\n"
        f"- Top 5 anomalous trace groups (cmdb_id | trace_name | details):\n  {top5_text}\n"
        f"- Frequency of anomalous groups per cmdb_id (top 10): see table below.\n"
        f"Notes: severity = max_value_in_window / p{thresh.upper()}; 'Inf' indicates p{thresh.upper()}==0 and max>0."
    )

# Display the concise summary string and the compact frequency table (top 10)
summary, freq_tbl
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  p95_value  count_anom            first_anom_ts  max_value_in_window  severity_ratio  longest_consecutive_anom_run
0   dockerA2            trace.self.duration_p95   0.015585           2  2021-03-09 19:32:00 CST             0.310500       19.923003                             1
1   dockerB2         trace.to_MG02.duration_p95   0.058072           2  2021-03-09 19:31:00 CST             0.591100       10.178656                             2
2       MG02   trace.from_dockerB2.duration_p95   0.058072           2  2021-03-09 19:31:00 CST             0.591100       10.178656                             2
3       MG01   trace.from_dockerB2.duration_p95   0.230795           2  2021-03-09 19:40:00 CST             1.012300        4.386144                             1
4   dockerB2         trace.to_MG01.duration_p95   0.230795           2  2021-03-09 19:40:00 CST             1.012300        4.386144                             1
5   dockerB1         trace.to_MG01.duration_p95   0.363025           2  2021-03-09 19:30:00 CST             1.015000        2.795951                             1
6       MG01   trace.from_dockerB1.duration_p95   0.363025           2  2021-03-09 19:30:00 CST             1.015000        2.795951                             1
7       MG01  trace.from_dockerA2.duration_mean   0.074368           2  2021-03-09 19:32:00 CST             0.205250        2.759933                             1
8   dockerA2        trace.to_MG01.duration_mean   0.074368           2  2021-03-09 19:32:00 CST             0.205250        2.759933                             1
9   dockerB1         trace.to_MG02.duration_p95   0.240547           2  2021-03-09 19:32:00 CST             0.625750        2.601357                             1
10      MG02   trace.from_dockerB1.duration_p95   0.240547           2  2021-03-09 19:32:00 CST             0.625750        2.601357                             1
11      MG02  trace.from_dockerA2.duration_mean   0.047815           2  2021-03-09 19:38:00 CST             0.091333        1.910149                             1
12  dockerA2        trace.to_MG02.duration_mean   0.047815           2  2021-03-09 19:38:00 CST             0.091333        1.910149                             1
13  dockerA2         trace.to_MG01.duration_p95   0.372940           2  2021-03-09 19:32:00 CST             0.682750        1.830723                             1
14      MG01   trace.from_dockerA2.duration_p95   0.372940           2  2021-03-09 19:32:00 CST             0.682750        1.830723                             1
15      MG02  trace.from_dockerA1.duration_mean   0.135992           2  2021-03-09 19:32:00 CST             0.232316        1.708309                             1
16  dockerA1        trace.to_MG02.duration_mean   0.135992           2  2021-03-09 19:32:00 CST             0.232316        1.708309                             1
17  dockerA2         trace.to_MG02.duration_p95   0.039292           2  2021-03-09 19:30:00 CST             0.067100        1.707705                             1
18      MG02   trace.from_dockerA2.duration_p95   0.039292           2  2021-03-09 19:30:00 CST             0.067100        1.707705                             1
19      MG01    trace.to_dockerB2.duration_mean   0.297084           2  2021-03-09 19:32:00 CST             0.486847        1.638750                             1, 255, ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'],     cmdb_id  anomalous_group_count
0      MG02                     42
1      MG01                     42
2      IG01                     18
3  Tomcat01                     18
4  dockerA1                     18
5  Tomcat02                     18
6  Tomcat03                     18
7  Tomcat04                     18
8  dockerB1                     18
9      IG02                     15, 'p95')```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing variables if present, else load files
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

try:
    df_error_logs
except NameError:
    df_error_logs = pd.read_csv('error_logs.csv')

# Ensure numeric 'value' in log
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(pd.Timestamp('2021-03-09 19:30:00'))
end_local   = tz.localize(pd.Timestamp('2021-03-09 20:00:00'))
start_utc_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_utc_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# Helper to compute group quantiles (global, BEFORE filtering)
def group_quantile(df, group_cols, q, col_name):
    return df.groupby(group_cols)['value'].agg(
        **{col_name: (lambda x, q=q: float(np.nanpercentile(x.dropna(), q)) if x.dropna().size>0 else np.nan)}
    ).reset_index()

# Compute global P95 and P90 for log groups
p95_logs = group_quantile(df_log, ['cmdb_id','log_name'], 95, 'p95_value')
p90_logs = group_quantile(df_log, ['cmdb_id','log_name'], 90, 'p90_value')
thresholds_logs = p95_logs.merge(p90_logs, on=['cmdb_id','log_name'], how='left')

# Merge thresholds into log rows (no filtering yet)
df_logm = df_log.merge(thresholds_logs, on=['cmdb_id','log_name'], how='left')

# Filter to incident window
df_log_win = df_logm[(df_logm['timestamp'] >= start_utc_ts) & (df_logm['timestamp'] <= end_utc_ts)].copy()

# Function to build anomaly summary for a given threshold column
def build_log_anom_summary(df_window, thresholds_df, threshold_col):
    if df_window.empty:
        return pd.DataFrame()
    mask = (~df_window[threshold_col].isna()) & (df_window['value'] >= df_window[threshold_col])
    anom_rows = df_window[mask].copy()
    if anom_rows.empty:
        return pd.DataFrame()
    # Aggregations
    grp_count = anom_rows.groupby(['cmdb_id','log_name'])['value'].count().rename('count_anom')
    grp_first = anom_rows.groupby(['cmdb_id','log_name'])['timestamp'].min().rename('first_anom_ts')
    grp_max = df_window.groupby(['cmdb_id','log_name'])['value'].max().rename('max_value_in_window')
    summary = pd.concat([grp_count, grp_first, grp_max], axis=1).reset_index()
    # attach threshold
    summary = summary.merge(thresholds_df[['cmdb_id','log_name', threshold_col]],
                            on=['cmdb_id','log_name'], how='left')
    # severity_ratio rules
    def compute_ratio(row):
        p = row[threshold_col]
        mx = row['max_value_in_window']
        if pd.isna(p):
            return np.nan
        if p == 0:
            if mx > 0:
                return np.inf
            else:
                return 1.0
        return float(mx / p)
    summary['severity_ratio'] = summary.apply(compute_ratio, axis=1)
    # format timestamp to UTC+8
    summary['first_anom_ts'] = pd.to_datetime(summary['first_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # round values
    summary[threshold_col] = summary[threshold_col].round(6)
    summary['max_value_in_window'] = summary['max_value_in_window'].round(6)
    # rename threshold column to p95_value/p90_value appropriately
    out_col = 'p95_value' if threshold_col == 'p95_value' else 'p90_value'
    summary = summary.rename(columns={threshold_col: out_col})
    # select requested columns
    summary = summary[['cmdb_id','log_name', out_col, 'count_anom', 'first_anom_ts', 'max_value_in_window', 'severity_ratio']]
    return summary

# Try P95 first
log_anom_p95 = build_log_anom_summary(df_log_win, thresholds_logs, 'p95_value')
used_threshold = 'p95'
final_log_anom = log_anom_p95.copy()

# If empty, use P90
if final_log_anom.empty:
    log_anom_p90 = build_log_anom_summary(df_log_win, thresholds_logs, 'p90_value')
    final_log_anom = log_anom_p90.copy()
    used_threshold = 'p90'

# Limit to top 20 by severity_ratio desc
if final_log_anom.empty:
    log_anom_top20 = final_log_anom  # empty
else:
    # Replace NaN severity with -inf so they go to bottom, but keep NaN visible
    log_anom_top20 = final_log_anom.sort_values(by='severity_ratio', ascending=False).head(20).reset_index(drop=True)

# --- error_logs.csv processing ---
df_err = df_error_logs.copy()
# Filter to incident window
df_err_win = df_err[(df_err['timestamp'] >= start_utc_ts) & (df_err['timestamp'] <= end_utc_ts)].copy()

# Overall total number of error_log messages in window
total_error_messages = int(df_err_win.shape[0])

# Per-cmdb_id aggregates: count_messages, earliest_message_ts, up to 5 example messages
if df_err_win.empty:
    err_group_top10 = pd.DataFrame(columns=['cmdb_id','count_messages','earliest_message_ts','example_messages'])
else:
    # group
    def sample_messages(s):
        # take up to 5 unique example messages in original order
        vals = s.dropna().astype(str).tolist()
        # keep order and unique
        seen = set()
        out = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                out.append(v)
            if len(out) >= 5:
                break
        return " | ".join(out)
    grp = df_err_win.groupby('cmdb_id').agg(
        count_messages = ('message','count'),
        earliest_message_ts = ('timestamp','min'),
        example_messages = ('message', sample_messages)
    ).reset_index()
    # format earliest ts
    grp['earliest_message_ts'] = pd.to_datetime(grp['earliest_message_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # sort and limit top 10
    err_group_top10 = grp.sort_values(by='count_messages', ascending=False).head(10).reset_index(drop=True)

# Final compact outputs
log_anom_top20, used_threshold, err_group_top10, total_error_messages

```
Out[5]:
```
```python
# Compose a concise plain-English summary of the log-based localization results
try:
    table = log_anom_top20    # compact table of log-group anomalies (top 20 by severity)
    thr = used_threshold
    err_table = err_group_top10
    total_err = total_error_messages
except NameError:
    # fallback: minimal recompute if variables missing
    import pandas as pd, numpy as np, pytz
    table = pd.DataFrame()
    thr = 'p95'
    err_table = pd.DataFrame()
    total_err = 0

if table is None or table.shape[0] == 0:
    summary = "No log-group anomalies detected in the incident window using P95/P90 thresholds."
else:
    n_groups = int(table.shape[0])
    # Build top-5 summary lines
    top_lines = []
    for r in table.head(5).itertuples(index=False):
        cmdb = getattr(r, 'cmdb_id')
        name = getattr(r, 'log_name')
        pval = getattr(r, table.columns[2])
        cnt = int(getattr(r, 'count_anom'))
        first = getattr(r, 'first_anom_ts')
        mx = float(getattr(r, 'max_value_in_window'))
        sev = getattr(r, 'severity_ratio')
        top_lines.append(f"{cmdb} | {name} | p{thr}={pval:.6g} | count={cnt} | first={first} | max={mx:.6g} | severity={sev:.6g}")
    top5_text = " ; ".join(top_lines)

    # error_logs summary
    if err_table is None or err_table.shape[0] == 0:
        err_summary = "No error_log messages found in the incident window."
    else:
        # show top entries of err_table compactly
        err_lines = []
        for er in err_table.itertuples(index=False):
            err_lines.append(f"{er.cmdb_id} ({er.count_messages} msgs, earliest={er.earliest_message_ts})")
        err_summary = f"Error log activity (top cmdb_id): " + "; ".join(err_lines)

    summary = (
        f"Log-based localization (window 2021-03-09 19:30:00 to 2021-03-09 20:00:00 UTC+8):\n"
        f"- Threshold used: global {thr.upper()} per log group.\n"
        f"- Log groups with breaches in the window: {n_groups} (top 20 shown).\n"
        f"- Top 5 anomalous log groups (cmdb_id | log_name | p{thr} | count_anom | first_anom_ts | max_value | severity_ratio):\n  {top5_text}\n"
        f"- Notes: Several groups show modest breaches (severity ~1.05–1.29). Some groups with p95==0 had max==0 (ratio=1).\n"
        f"- Error logs: total messages in window = {int(total_err)}. {err_summary}\n"
        f"Interpretation: The strongest log signals are GC pause metrics on Tomcat hosts (Tomcat02, Tomcat01, Tomcat03) and some row_count increases on apache/Tomcat. No error-log messages were present in the window."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  p95_value  count_anom            first_anom_ts  max_value_in_window  severity_ratio
0   Tomcat02  log.gc_pause_max     0.1160           1  2021-03-09 19:40:00 CST                 0.15        1.293103
1   Tomcat01  log.gc_pause_max     0.0935           1  2021-03-09 19:46:00 CST                 0.11        1.176471
2   Tomcat03  log.gc_pause_max     0.1150           1  2021-03-09 19:42:00 CST                 0.13        1.130435
3   apache02     log.row_count   503.5000           2  2021-03-09 19:53:00 CST               545.00        1.082423
4   Tomcat03     log.row_count   242.5000           2  2021-03-09 19:54:00 CST               260.00        1.072165
5   Tomcat04  log.gc_pause_max     0.0850           1  2021-03-09 19:55:00 CST                 0.09        1.058824
6   apache01     log.row_count   500.0000           2  2021-03-09 19:53:00 CST               526.00        1.052000
7   Tomcat01     log.row_count   263.5000           2  2021-03-09 19:54:00 CST               276.00        1.047438
8   Tomcat04     log.row_count   242.0000           2  2021-03-09 19:53:00 CST               247.00        1.020661
9   Tomcat02     log.row_count     1.0000          18  2021-03-09 19:31:00 CST                 1.00        1.000000
10  Tomcat01   log.error_count     0.0000          31  2021-03-09 19:30:00 CST                 0.00        1.000000
11  Tomcat02   log.error_count     0.0000          18  2021-03-09 19:31:00 CST                 0.00        1.000000
12  Tomcat03   log.error_count     0.0000          31  2021-03-09 19:30:00 CST                 0.00        1.000000
13  Tomcat04   log.error_count     0.0000          31  2021-03-09 19:30:00 CST                 0.00        1.000000
14  apache01   log.error_count     0.0000          31  2021-03-09 19:30:00 CST                 0.00        1.000000
15  apache02   log.error_count     0.0000          31  2021-03-09 19:30:00 CST                 0.00        1.000000, 'p95', Empty DataFrame
Columns: [cmdb_id, count_messages, earliest_message_ts, example_messages]
Index: [], 0)```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd
import pytz

# Load metric.csv if not already loaded
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure numeric 'value'
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(pd.Timestamp('2021-03-09 19:30:00'))
end_local   = tz.localize(pd.Timestamp('2021-03-09 20:00:00'))
start_utc_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_utc_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# Target cmdb_id
tomcat_id = 'Tomcat02'

# 1) Distinct kpi_name for Tomcat02 (full series)
tomcat_kpis = pd.Series(sorted(df_metric[df_metric['cmdb_id'] == tomcat_id]['kpi_name'].dropna().unique()),
                        name='kpi_name')

# Helper: compute global quantile per group BEFORE filtering
def compute_global_quantile(df, q, group_cols, out_name):
    agg = df.groupby(group_cols)['value'].agg(
        **{out_name: (lambda x, q=q: float(np.nanpercentile(x.dropna(), q)) if x.dropna().size>0 else np.nan)}
    ).reset_index()
    return agg

# 2) Compute global P95 thresholds for all groups
p95_thresholds = compute_global_quantile(df_metric, 95, ['cmdb_id','kpi_name'], 'p95_value')

# 3) Filter metric rows to incident window and Tomcat02
df_win = df_metric[(df_metric['cmdb_id'] == tomcat_id) & 
                   (df_metric['timestamp'] >= start_utc_ts) & 
                   (df_metric['timestamp'] <= end_utc_ts)].copy()

# Merge P95 thresholds into the window rows (thresholds computed globally)
df_win = df_win.merge(p95_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Helper: longest consecutive-minute run
def longest_consecutive_minute_run(ts_array):
    if len(ts_array) == 0:
        return 0
    ts = np.sort(np.unique(ts_array.astype(np.int64)))
    if ts.size == 1:
        return 1
    run_len = 1
    runs = []
    for d in np.diff(ts):
        if d == 60:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return int(np.max(runs))

# Function to build Tomcat summary given a threshold column present in df_win
def build_summary_from_window(df_window, threshold_col):
    if df_window.empty:
        return pd.DataFrame(), pd.DataFrame()  # summary, anom_rows
    mask = (~df_window[threshold_col].isna()) & (df_window['value'] >= df_window[threshold_col])
    anom_rows = df_window[mask].copy()
    if anom_rows.empty:
        return pd.DataFrame(), anom_rows
    # Aggregations
    grp_count = anom_rows.groupby(['cmdb_id','kpi_name'])['value'].count().rename('count_anom')
    grp_first = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].min().rename('first_anom_ts')
    grp_last = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].max().rename('last_anom_ts')
    grp_max_window = df_window.groupby(['cmdb_id','kpi_name'])['value'].max().rename('max_value_in_window')
    summary = pd.concat([grp_count, grp_first, grp_last, grp_max_window], axis=1).reset_index()
    # attach threshold
    summary = summary.merge(df_window[['cmdb_id','kpi_name', threshold_col]].drop_duplicates(),
                            on=['cmdb_id','kpi_name'], how='left')
    # compute severity_ratio with special rules
    def compute_ratio(p, mx):
        if pd.isna(p):
            return np.nan
        if p == 0:
            if mx > 0:
                return np.inf
            else:
                return 1.0
        return float(mx / p)
    summary['severity_ratio'] = summary.apply(lambda r: compute_ratio(r[threshold_col], r['max_value_in_window']), axis=1)
    # compute longest consecutive run per group
    runs = []
    grouped = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp']
    for cmdb, kpi in summary[['cmdb_id','kpi_name']].itertuples(index=False):
        try:
            arr = grouped.get_group((cmdb, kpi)).values
        except Exception:
            arr = anom_rows[(anom_rows['cmdb_id']==cmdb) & (anom_rows['kpi_name']==kpi)]['timestamp'].values
        runs.append(longest_consecutive_minute_run(arr))
    summary['longest_consecutive_anom_run'] = runs
    # format timestamps to UTC+8 strings for display
    summary['first_anom_ts_str'] = pd.to_datetime(summary['first_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    summary['last_anom_ts_str'] = pd.to_datetime(summary['last_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # round numeric
    summary[threshold_col] = summary[threshold_col].round(6)
    summary['max_value_in_window'] = summary['max_value_in_window'].round(6)
    # significant_breach flag
    summary['significant_breach'] = summary['severity_ratio'].apply(lambda x: True if (pd.notna(x) and x >= 1.5) else False)
    # select and order columns for output
    out_col = 'p95_value' if threshold_col == 'p95_value' else 'p90_value'
    summary = summary.rename(columns={threshold_col: out_col})
    summary_out = summary[['cmdb_id','kpi_name', out_col, 'count_anom', 'first_anom_ts_str', 'last_anom_ts_str',
                           'max_value_in_window','severity_ratio','longest_consecutive_anom_run','significant_breach']].rename(
        columns={'first_anom_ts_str':'first_anom_ts', 'last_anom_ts_str':'last_anom_ts'}
    )
    return summary_out, anom_rows

# 4) Build summary using P95 first
summary_p95, anom_rows_p95 = build_summary_from_window(df_win, 'p95_value')
used_threshold = 'p95'
final_summary = summary_p95.copy()
anom_rows_global = anom_rows_p95.copy()

# 5) If no P95 breaches, compute P90 globally and retry
if final_summary.empty:
    p90_thresholds = compute_global_quantile(df_metric, 90, ['cmdb_id','kpi_name'], 'p90_value')
    # Merge P90 into df_win
    df_win = df_win.merge(p90_thresholds, on=['cmdb_id','kpi_name'], how='left')
    summary_p90, anom_rows_p90 = build_summary_from_window(df_win, 'p90_value')
    final_summary = summary_p90.copy()
    anom_rows_global = anom_rows_p90.copy()
    used_threshold = 'p90'

# Prepare final outputs
if final_summary.empty:
    tomcat_result = final_summary
    significant_count = 0
    earliest_first_anom_ts = None
else:
    tomcat_result = final_summary.sort_values(by='severity_ratio', ascending=False).head(20).reset_index(drop=True)
    significant_count = int(tomcat_result['significant_breach'].sum())
    # Earliest first_anom_ts among all Tomcat02 anomalous groups (use numeric anom_rows_global if available)
    if not anom_rows_global.empty:
        earliest_ts = int(anom_rows_global['timestamp'].min())
        earliest_first_anom_ts = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest_first_anom_ts = None

# Final compact outputs
tomcat_kpis, tomcat_result, significant_count, earliest_first_anom_ts, used_threshold

```
Out[6]:
```
```python
# Compose a concise plain-English summary of the Tomcat02-focused fault identification
try:
    kpi_series = tomcat_kpis
    df_top = tomcat_result
    sig_count = significant_count
    earliest = earliest_first_anom_ts
    thr = used_threshold
except NameError:
    # fallback in case variables are missing
    import pandas as pd, numpy as np
    kpi_series = pd.Series([], name='kpi_name')
    df_top = pd.DataFrame()
    sig_count = 0
    earliest = None
    thr = 'p95'

if df_top is None or df_top.shape[0] == 0:
    summary = ("No Tomcat02 metric-group anomalies detected in the incident window using P95/P90 thresholds.")
else:
    total_kpis = int(kpi_series.shape[0])
    shown_groups = int(df_top.shape[0])
    # Build top-5 description lines
    lines = []
    for r in df_top.head(5).itertuples(index=False):
        cmdb = r.cmdb_id
        kpi = r.kpi_name
        pval = getattr(r, 'p95_value') if 'p95_value' in df_top.columns else getattr(r, 'p90_value')
        cnt = int(r.count_anom)
        first = r.first_anom_ts
        last = r.last_anom_ts if 'last_anom_ts' in df_top.columns else ''
        mx = float(r.max_value_in_window)
        sev = r.severity_ratio
        run = int(r.longest_consecutive_anom_run) if 'longest_consecutive_anom_run' in df_top.columns else 0
        sig = bool(r.significant_breach) if 'significant_breach' in df_top.columns else False
        lines.append(f"{kpi} | p{thr}={pval:.6g} | count={cnt} | first={first} | max={mx:.6g} | severity={sev:.6g} | run={run} | sig={sig}")
    top5_text = " ; ".join(lines)

    summary = (
        f"Tomcat02 fault-identification (window 2021-03-09 19:30:00 to 2021-03-09 20:00:00 UTC+8):\n"
        f"- Distinct Tomcat02 KPIs in full metrics: {total_kpis}.\n"
        f"- Threshold used: global {thr.upper()} per KPI (computed on full series).\n"
        f"- Anomalous Tomcat02 KPI groups (showing top {shown_groups} by severity): {shown_groups} groups.\n"
        f"- Number of Tomcat02 groups with significant breach (severity_ratio >= 1.5): {sig_count}.\n"
        f"- Earliest anomaly occurrence on Tomcat02 (candidate root time): {earliest}.\n"
        f"- Top 5 anomalous KPIs (kpi_name | p{thr} | count_anom | first_anom_ts | max_value | severity_ratio | longest_run | significant):\n  {top5_text}\n"
        f"Interpretation: The strongest signals on Tomcat02 are network/TCP CLOSE-WAIT (p{thr}=0, max>0 → severity=Inf) and high per-CPU utilization (e.g., CPU-0 SingleCpuUtil with severity ~1.76). Several other KPIs show modest breaches (severity ~1.02–1.33). These are the primary Tomcat02 candidates to investigate, with the earliest anomalous timestamp at the listed time."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(0                        OSLinux-CPU_CPU-0_SingleCpuUtil
1                        OSLinux-CPU_CPU-0_SingleCpuidle
2                        OSLinux-CPU_CPU-1_SingleCpuUtil
3                        OSLinux-CPU_CPU-1_SingleCpuidle
4                        OSLinux-CPU_CPU-2_SingleCpuUtil
                             ...                        
104    Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...
105    Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...
106    Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...
107    Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...
108    Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...
Name: kpi_name, Length: 109, dtype: object,      cmdb_id                                           kpi_name     p95_value  count_anom            first_anom_ts             last_anom_ts  max_value_in_window  severity_ratio  longest_consecutive_anom_run  significant_breach
0   Tomcat02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00          21  2021-03-09 19:30:00 CST  2021-03-09 19:59:00 CST         1.000000e+00             inf                             3                True
1   Tomcat02                             OSLinux-CPU_CPU_CPUWio  1.580000e-01           2  2021-03-09 19:32:00 CST  2021-03-09 19:38:00 CST         3.369000e-01        2.132278                             1                True
2   Tomcat02                    OSLinux-CPU_CPU-0_SingleCpuUtil  3.265617e+01           1  2021-03-09 19:41:00 CST  2021-03-09 19:41:00 CST         5.732090e+01        1.755285                             1                True
3   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  3.000000e+00           3  2021-03-09 19:32:00 CST  2021-03-09 19:51:00 CST         4.000000e+00        1.333333                             1               False
4   Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps  2.500200e-01           1  2021-03-09 19:56:00 CST  2021-03-09 19:56:00 CST         3.167000e-01        1.266699                             1               False
5   Tomcat02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  3.120000e+00           1  2021-03-09 19:56:00 CST  2021-03-09 19:56:00 CST         3.600000e+00        1.153846                             1               False
6   Tomcat02    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  2.451605e+05           1  2021-03-09 19:53:00 CST  2021-03-09 19:53:00 CST         2.824750e+05        1.152204                             1               False
7   Tomcat02        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  5.532000e+02           1  2021-03-09 19:54:00 CST  2021-03-09 19:54:00 CST         6.000000e+02        1.084599                             1               False
8   Tomcat02       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  6.055000e+02           1  2021-03-09 19:54:00 CST  2021-03-09 19:54:00 CST         6.550000e+02        1.081751                             1               False
9   Tomcat02    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil  2.940000e-03           1  2021-03-09 19:38:00 CST  2021-03-09 19:38:00 CST         3.100000e-03        1.054422                             1               False
10  Tomcat02    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  5.649500e+02           1  2021-03-09 19:33:00 CST  2021-03-09 19:33:00 CST         5.820000e+02        1.030180                             1               False
11  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory  2.703986e+09           1  2021-03-09 19:33:00 CST  2021-03-09 19:33:00 CST         2.769329e+09        1.024166                             1               False
12  Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps  9.467000e-01           1  2021-03-09 19:56:00 CST  2021-03-09 19:56:00 CST         9.667000e-01        1.021126                             1               False
13  Tomcat02                    OSLinux-CPU_CPU-2_SingleCpuUtil  3.436580e+00           1  2021-03-09 19:41:00 CST  2021-03-09 19:41:00 CST         3.509100e+00        1.021102                             1               False
14  Tomcat02                         OSLinux-CPU_CPU_CPUSysTime  6.777000e-01           2  2021-03-09 19:35:00 CST  2021-03-09 19:45:00 CST         6.901000e-01        1.018297                             1               False
15  Tomcat02                            OSLinux-CPU_CPU_CPULoad  1.240000e+00           2  2021-03-09 19:45:00 CST  2021-03-09 19:51:00 CST         1.260000e+00        1.016129                             1               False
16  Tomcat02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite  1.202666e+01           1  2021-03-09 19:48:00 CST  2021-03-09 19:48:00 CST         1.213330e+01        1.008867                             1               False
17  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory  2.071952e+09           1  2021-03-09 19:36:00 CST  2021-03-09 19:36:00 CST         2.087812e+09        1.007655                             1               False
18  Tomcat02                         OSLinux-CPU_CPU_CPUCpuUtil  2.718514e+01           1  2021-03-09 19:33:00 CST  2021-03-09 19:33:00 CST         2.736650e+01        1.006671                             1               False
19  Tomcat02                    OSLinux-CPU_CPU-1_SingleCpuidle  9.761708e+01           1  2021-03-09 19:38:00 CST  2021-03-09 19:38:00 CST         9.807020e+01        1.004642                             1               False, 3, '2021-03-09 19:30:00 CST', 'p95')```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing df_trace if present, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure numeric 'value'
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(pd.Timestamp('2021-03-09 19:30:00'))
end_local   = tz.localize(pd.Timestamp('2021-03-09 20:00:00'))
start_utc_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_utc_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# Target cmdb_id
target_cmdb = 'Tomcat02'

# Helper: compute global quantile per group BEFORE filtering
def compute_group_quantile(df, q):
    name = f'p{int(q)}_value'
    agg = df.groupby(['cmdb_id','trace_name'])['value'].agg(
        **{name: (lambda x, q=q: float(np.nanpercentile(x.dropna(), q)) if x.dropna().size>0 else np.nan)}
    ).reset_index()
    return agg

# Helper: longest consecutive-minute run from epoch-second timestamps
def longest_consecutive_minute_run(ts_array):
    if len(ts_array) == 0:
        return 0
    ts = np.sort(np.unique(ts_array.astype(np.int64)))
    if ts.size == 1:
        return 1
    runs = []
    run_len = 1
    diffs = np.diff(ts)
    for d in diffs:
        if d == 60:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return int(np.max(runs))

# Compute global P95 and P90 thresholds (before any filtering)
p95_all = compute_group_quantile(df_trace, 95)
p90_all = compute_group_quantile(df_trace, 90)

# Filter trace rows to incident window and target cmdb_id
df_win = df_trace[(df_trace['timestamp'] >= start_utc_ts) & (df_trace['timestamp'] <= end_utc_ts) & (df_trace['cmdb_id'] == target_cmdb)].copy()

# Prepare thresholds for target cmdb
p95_target = p95_all[p95_all['cmdb_id'] == target_cmdb].rename(columns={'p95_value':'p95_value'})
p90_target = p90_all[p90_all['cmdb_id'] == target_cmdb].rename(columns={'p90_value':'p90_value'})

# Merge P95 into window rows
df_win = df_win.merge(p95_target[['cmdb_id','trace_name','p95_value']], on=['cmdb_id','trace_name'], how='left')

# Function to build summary given threshold column name in rows (e.g., 'p95_value' or 'p90_value')
def build_summary(df_window, threshold_col):
    if df_window.empty:
        return pd.DataFrame()
    # mask rows where value >= threshold (and threshold not NaN)
    mask = (~df_window[threshold_col].isna()) & (df_window['value'] >= df_window[threshold_col])
    anom_rows = df_window[mask].copy()
    if anom_rows.empty:
        return pd.DataFrame()
    # Aggregations per trace_name (cmdb_id is constant = target_cmdb)
    grp = anom_rows.groupby(['cmdb_id','trace_name'])
    count_anom = grp.size().rename('count_anom')
    first_anom_ts = grp['timestamp'].min().rename('first_anom_ts')
    # max_value_in_window should consider all rows in window for the group
    max_in_window = df_window.groupby(['cmdb_id','trace_name'])['value'].max().rename('max_value_in_window')
    summary = pd.concat([count_anom, first_anom_ts, max_in_window], axis=1).reset_index()
    # attach global threshold value per group
    # threshold values exist in df_window (merged), but ensure unique
    thr_vals = df_window[['cmdb_id','trace_name', threshold_col]].drop_duplicates(subset=['cmdb_id','trace_name'])
    summary = summary.merge(thr_vals, on=['cmdb_id','trace_name'], how='left')
    # compute severity_ratio with special rules
    def comp_ratio(p, mx):
        if pd.isna(p):
            return np.nan
        if p == 0:
            if mx > 0:
                return np.inf
            else:
                return 1.0
        return float(mx / p)
    summary['severity_ratio'] = summary.apply(lambda r: comp_ratio(r[threshold_col], r['max_value_in_window']), axis=1)
    # longest consecutive run per group using anom_rows timestamps
    runs = []
    grouped_ts = anom_rows.groupby(['cmdb_id','trace_name'])['timestamp']
    for cmdb, trace_name in summary[['cmdb_id','trace_name']].itertuples(index=False):
        try:
            arr = grouped_ts.get_group((cmdb, trace_name)).values
        except Exception:
            arr = anom_rows[(anom_rows['cmdb_id']==cmdb) & (anom_rows['trace_name']==trace_name)]['timestamp'].values
        runs.append(longest_consecutive_minute_run(arr))
    summary['longest_consecutive_anom_run'] = runs
    # format first_anom_ts to UTC+8 strings
    summary['first_anom_ts'] = pd.to_datetime(summary['first_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # round numeric for compactness
    summary[threshold_col] = summary[threshold_col].round(6)
    summary['max_value_in_window'] = summary['max_value_in_window'].round(6)
    # select and order requested columns; rename threshold column to p95_value/p90_value
    out_col = 'p95_value' if threshold_col == 'p95_value' else 'p90_value'
    summary = summary.rename(columns={threshold_col: out_col})
    summary = summary[['cmdb_id','trace_name', out_col, 'count_anom', 'first_anom_ts', 'max_value_in_window', 'severity_ratio', 'longest_consecutive_anom_run']]
    return summary

# Try using P95 first
used_threshold = 'p95'
summary_p95 = build_summary(df_win, 'p95_value')

# If no groups breach P95, recompute/merge P90 and retry
if summary_p95.empty:
    # Merge P90 into df_win (ensure using global P90)
    df_win = df_win.drop(columns=[c for c in df_win.columns if c == 'p95_value'], errors='ignore')
    df_win = df_win.merge(p90_target[['cmdb_id','trace_name','p90_value']], on=['cmdb_id','trace_name'], how='left')
    used_threshold = 'p90'
    summary_p90 = build_summary(df_win, 'p90_value')
    final_summary = summary_p90.copy()
else:
    final_summary = summary_p95.copy()

# Prepare outputs: top 20 sorted by severity_ratio desc
if final_summary.empty:
    final_top20 = final_summary  # empty
    total_groups_with_anom = 0
    earliest_first_anom_ts = None
else:
    final_top20 = final_summary.sort_values(by='severity_ratio', ascending=False).head(20).reset_index(drop=True)
    total_groups_with_anom = int(final_summary.shape[0])
    # earliest first_anom_ts among Tomcat02 anomalous trace groups
    # parse back to timestamps for min
    earliest_str = final_top20['first_anom_ts'].min()
    earliest_first_anom_ts = earliest_str

# Final compact outputs
final_top20, total_groups_with_anom, earliest_first_anom_ts, used_threshold

```
Out[7]:
```
```python
# Compose a concise plain-English summary of the Tomcat02 trace-anomaly scan
try:
    df_top = final_top20
    total = total_groups_with_anom
    earliest = earliest_first_anom_ts
    thresh = used_threshold
except NameError:
    df_top = None
    total = 0
    earliest = None
    thresh = 'p95'

if df_top is None or df_top.shape[0] == 0:
    summary = "No Tomcat02 trace-group anomalies detected in the incident window using P95/P90 thresholds."
else:
    top_lines = []
    for r in df_top.head(5).itertuples(index=False):
        # columns: cmdb_id, trace_name, p95_value/p90_value, count_anom, first_anom_ts, max_value_in_window, severity_ratio, longest_consecutive_anom_run
        trace_name = r.trace_name
        pval = getattr(r, df_top.columns[2])
        cnt = int(r.count_anom)
        first = r.first_anom_ts
        mx = float(r.max_value_in_window)
        sev = r.severity_ratio
        run = int(r.longest_consecutive_anom_run)
        top_lines.append(f"{trace_name} | p{thresh}={pval:.6g} | count={cnt} | first={first} | max={mx:.6g} | severity={sev:.6g} | run={run}")
    top5_text = " ; ".join(top_lines)

    summary = (
        f"Tomcat02 trace-anomaly summary (window 2021-03-09 19:30:00 to 2021-03-09 20:00:00 UTC+8):\n"
        f"- Threshold used: global {thresh.upper()} per trace group.\n"
        f"- Trace groups with anomalies in window: {total} groups.\n"
        f"- Earliest anomalous timestamp among Tomcat02 traces: {earliest}.\n"
        f"- Top 5 anomalous traces (trace_name | p{thresh} | count_anom | first_anom_ts | max_value | severity_ratio | longest_run):\n  {top5_text}\n"
        f"Interpretation: Tomcat02 shows multiple trace-level degradations (latency and row_count increases) with modest severity (severity ~1.05–1.31). The highest relative increase is in trace.to_MG01.duration_mean (severity ~1.31)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                     trace_name    p95_value  count_anom            first_anom_ts  max_value_in_window  severity_ratio  longest_consecutive_anom_run
0   Tomcat02    trace.to_MG01.duration_mean     0.314193           2  2021-03-09 19:32:00 CST             0.410258        1.305751                             1
1   Tomcat02        trace.self.duration_p95     0.061625           2  2021-03-09 19:31:00 CST             0.077100        1.251116                             1
2   Tomcat02  trace.from_IG02.duration_mean     0.308909           2  2021-03-09 19:32:00 CST             0.380657        1.232262                             1
3   Tomcat02   trace.from_IG02.duration_p95     1.707215           2  2021-03-09 19:32:00 CST             2.092400        1.225622                             1
4   Tomcat02           trace.self.row_count  1792.600000           2  2021-03-09 19:53:00 CST          2069.000000        1.154189                             2
5   Tomcat02        trace.to_MG01.row_count   137.550000           2  2021-03-09 19:53:00 CST           158.000000        1.148673                             2
6   Tomcat02  trace.from_IG01.duration_mean     0.404645           2  2021-03-09 19:32:00 CST             0.459886        1.136517                             1
7   Tomcat02     trace.to_MG02.duration_p95     1.845995           2  2021-03-09 19:31:00 CST             2.069000        1.120805                             1
8   Tomcat02    trace.to_MG02.duration_mean     0.318757           2  2021-03-09 19:31:00 CST             0.347174        1.089150                             1
9   Tomcat02       trace.self.duration_mean     0.054766           2  2021-03-09 19:32:00 CST             0.059498        1.086403                             1
10  Tomcat02      trace.from_IG02.row_count   135.200000           2  2021-03-09 19:32:00 CST           146.000000        1.079882                             1
11  Tomcat02      trace.from_IG01.row_count   139.950000           2  2021-03-09 19:53:00 CST           151.000000        1.078957                             2
12  Tomcat02     trace.to_MG01.duration_p95     2.011467           2  2021-03-09 19:32:00 CST             2.125750        1.056815                             1
13  Tomcat02   trace.from_IG01.duration_p95     2.083072           2  2021-03-09 19:32:00 CST             2.131800        1.023392                             1
14  Tomcat02        trace.to_MG02.row_count   136.200000           2  2021-03-09 19:53:00 CST           139.000000        1.020558                             2
15  Tomcat02      trace.from_root.row_count     6.000000           1  2021-03-09 19:30:00 CST             6.000000        1.000000                             1
16  Tomcat02  trace.from_root.duration_mean     0.016833           1  2021-03-09 19:30:00 CST             0.016833        1.000000                             1
17  Tomcat02   trace.from_root.duration_p95     0.071250           1  2021-03-09 19:30:00 CST             0.071250        1.000000                             1, 18, '2021-03-09 19:30:00 CST', 'p95')```
```